In [18]:
import pandas as pd

df = pd.read_csv("./traces-td1 copy.csv")
ref = pd.read_csv("./reference-td1.csv")

In [19]:
df.columns

Index(['cell_id', 'provider', 'model_requested', 'model_returned', 'case_id',
       'question', 'prompt_sha256', 'sql', 'value', 'reference', 'verdict',
       'verdict_detail', 'http_status', 'error', 'latency_ms', 'tokens_prompt',
       'tokens_completion', 'tokens_reasoning', 'cost_native', 'attempts',
       'date_utc', 'nature'],
      dtype='object')

Finding Candidate Questions

In [23]:
# Conserver uniquement les réponses incorrectes
wrong = df[
    df["verdict"].astype(str).str.strip().str.lower() == "incorrect"
].copy()

# Normaliser les valeurs pour éviter les différences d'espaces
wrong["value_normalized"] = wrong["value"].astype(str).str.strip()

# Identifier les valeurs identiques proposées par au moins
# deux modèles différents pour une même question
candidates = (
    wrong.groupby(["case_id", "value_normalized"])
    .agg(
        models=("model_returned", lambda x: sorted(set(x.dropna()))),
        number_of_models=("model_returned", "nunique"),
        questions=("question", "first"),
        reference=("reference", "first"),
    )
    .reset_index()
)

# Garder uniquement les consensus de deux modèles minimum
candidates = candidates[candidates["number_of_models"] >= 2].copy()


display(candidates)

,case_id,value_normalized,models,number_of_models,questions,reference
0,C02,0.0,"[dots-studio/dots-3-note-preview:free, openai/...",2,"What is the total revenue excluding VAT, in eu...",590320
3,C09,0.0,"[cohere/north-mini-code:free, qwen/qwen3.8-27b]",2,"What is the total revenue excluding VAT, in eu...",487980
15,C19,0.0,"[cohere/north-mini-code:free, openai/gpt-oss-1...",2,"What is the revenue excluding VAT, in euro cen...",1990


# C19

## Identification and quorum verdict.

In [ ]:
c19 = df[df["case_id"] == "C19"].copy()

for _, row in c19.iterrows():
    print("\n" + "=" * 70)
    print("MODEL :", row["model_returned"])
    print("VALUE :", row["value"])
    print("REFERENCE :", row["reference"])
    print("VERDICT :", row["verdict"])
    print("\nSQL :")
    print(row["sql"])

In [32]:
candidates["models"][15]

['cohere/north-mini-code:free', 'openai/gpt-oss-120b']

In [34]:
c19[["model_returned","value","reference","verdict"]]

,model_returned,value,reference,verdict
63,openai/gpt-oss-120b,0.0,1990,incorrect
64,openai/gpt-oss-20b,1990.0,1990,correct
65,cohere/north-mini-code:free,0.0,1990,incorrect
66,qwen/qwen3.8-27b,83580.0,1990,incorrect
67,dots-studio/dots-3-note-preview:free,1990.0,1990,correct



The two models that returned an incorrect answer for C19 are `cohere/north-mini-code:free` and `openai/gpt-oss-120b`. Both models returned the same wrong value: **0**, whereas the independent reference value is **1,990**.

**Fixed 2-of-3 panel**

Both models belong to the fixed panel, which also includes `qwen/qwen3.8-27b`. The votes are distributed as follows:

- 0: 2 votes
- 83,580: 1 vote

Since the threshold is 2 out of 3, the incorrect value of **0 wins the fixed panel**.

**Full 3-of-5 quorum**

Across all five models, the votes are distributed as follows:

- 0: 2 votes
- 1,990: 2 votes
- 83,580: 1 vote

Since the threshold is 3 out of 5, no value reaches the required threshold. Therefore, the incorrect value of **0 does not win the full quorum**.

This case illustrates how the fixed 2-of-3 panel can accept a shared incorrect answer, while the larger 3-of-5 quorum prevents that incorrect answer from being validated.

## Faulty SQL clause diagnosis

In [39]:
north = c19[c19["model_returned"] == "cohere/north-mini-code:free"]

print(north["sql"].values[0])

SELECT COALESCE(SUM(o.quantity_product * p.price), 0) AS revenue_cents
FROM orders o
JOIN product p ON o.product_id = p.product_id
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.product_id = 'd'
  AND o.date >= '2026-09-01 00:00:00' AND o.date < '2026-10-01 00:00:00'
  AND c.birth_date <= date('2026-09-25', '-' || CAST((strftime('%Y-%m-%d', '2026-09-25') - strftime('%Y-%m-%d', c.birth_date)) / 365 AS INTEGER) || ' years')
  AND c.birth_date > date('2026-09-25', '-' || CAST((strftime('%Y-%m-%d', '2026-09-25') - strftime('%Y-%m-%d', c.birth_date)) / 365 + 1 AS INTEGER) || ' years');


The problem here: 
```sql 
(strftime('%Y-%m-%d', '2026-09-25') - strftime('%Y-%m-%d', c.birth_date)) / 365
```

This `strftime(...)` produces strings, not numbers. Therefore the substraction is substracting two strings that are implicitly casted into integers. Therefore it first calculates the difference in **years** then divides it by 365.

The correct syntax is:
```sql
c.birth_date > '2000-09-25'
AND c.birth_date <= '2001-09-25'```

In [41]:
gpt = c19[c19["model_returned"] == "openai/gpt-oss-120b"]

print(gpt["sql"].values[0])

SELECT COALESCE(SUM(o.quantity_product * p.price), 0) AS revenue_cents
FROM orders o
JOIN product p ON o.product_id = p.product_id
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.product_id = 'd'
  AND o.date >= '2026-09-01 00:00:00'
  AND o.date < '2026-10-01 00:00:00'
  AND c.birth_date = date('2026-09-25', '-25 years');


The problem is 

```sql
AND c.birth_date = date('2026-09-25', '-25 years');
```

This line tells the request to only select people born on the day exactly 25 years before `2026-09-25`. Therefore it only selects people born on `2001-09-25`.

The correct syntax would be:
```sql
c.birth_date > '2000-09-25'
AND c.birth_date <= '2001-09-25'
```

## Delta diagnostic SQL

In [ ]:
SELECT
    COUNT(DISTINCT c.customer_id) AS customer_count,
    COALESCE(SUM(o.quantity_product * p.price), 0) AS revenue_cents
FROM orders o
JOIN product p ON o.product_id = p.product_id
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.product_id = 'd'
  AND o.date >= '2026-09-01 00:00:00'
  AND o.date < '2026-10-01 00:00:00'
  AND c.birth_date > '2000-09-25'
  AND c.birth_date <= '2001-09-25';

**Output**

customer_count	| revenue_cents	
:---|:---
1|1990

In [ ]:
SELECT
    c.customer_id,
    c.birth_date,
    o.date,
    o.quantity_product,
    p.price,
    o.quantity_product * p.price AS revenue_cents
FROM orders o
JOIN product p ON o.product_id = p.product_id
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.product_id = 'd'
  AND o.date >= '2026-09-01 00:00:00'
  AND o.date < '2026-10-01 00:00:00'
  AND c.birth_date > '2000-09-25'
  AND c.birth_date <= '2001-09-25'
ORDER BY c.customer_id, o.date;

**Output**

customer_id|birth_date| date| quantity_product| price| revenue_cents
:---|:---|:---|:---|:---|:---|
C002|2001-09-24|2026-09-07 12:51:43|1|1990| 1990

In [ ]:
SELECT
    'Correct age condition' AS diagnostic,
    COALESCE(SUM(o.quantity_product * p.price), 0) AS revenue_cents
FROM orders o
JOIN product p ON o.product_id = p.product_id
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.product_id = 'd'
  AND o.date >= '2026-09-01 00:00:00'
  AND o.date < '2026-10-01 00:00:00'
  AND c.birth_date > '2000-09-25'
  AND c.birth_date <= '2001-09-25'

UNION ALL

SELECT
    'GPT-OSS condition' AS diagnostic,
    COALESCE(SUM(o.quantity_product * p.price), 0) AS revenue_cents
FROM orders o
JOIN product p ON o.product_id = p.product_id
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.product_id = 'd'
  AND o.date >= '2026-09-01 00:00:00'
  AND o.date < '2026-10-01 00:00:00'
  AND c.birth_date = date('2026-09-25', '-25 years');

**Output**
Correct age condition | GPT-OSS condition 
:---|:---
1990|0

In [ ]:
SELECT
    'Correct age condition' AS diagnostic,
    COALESCE(SUM(o.quantity_product * p.price), 0) AS revenue_cents
FROM orders o
JOIN product p ON o.product_id = p.product_id
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.product_id = 'd'
  AND o.date >= '2026-09-01 00:00:00'
  AND o.date < '2026-10-01 00:00:00'
  AND c.birth_date > '2000-09-25'
  AND c.birth_date <= '2001-09-25'

UNION ALL

SELECT
    'North Mini condition' AS diagnostic,
    COALESCE(SUM(o.quantity_product * p.price), 0) AS revenue_cents
FROM orders o
JOIN product p ON o.product_id = p.product_id
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.product_id = 'd'
  AND o.date >= '2026-09-01 00:00:00'
  AND o.date < '2026-10-01 00:00:00'
  AND c.birth_date <= date(
      '2026-09-25',
      '-' || CAST(
          (strftime('%Y-%m-%d', '2026-09-25')
          - strftime('%Y-%m-%d', c.birth_date)) / 365
          AS INTEGER
      ) || ' years'
  )
  AND c.birth_date > date(
      '2026-09-25',
      '-' || CAST(
          (strftime('%Y-%m-%d', '2026-09-25')
          - strftime('%Y-%m-%d', c.birth_date)) / 365 + 1
          AS INTEGER
      ) || ' years'
  );

**Output**
Correct age condition | North Mini condition 
:---|:---
1990|0

## C19 Conclusion

The correct revenue excluding VAT for product d is 1,990 euro cents. The SQL diagnostic confirms that the correct age interval returns one matching customer (C002) and revenue of 1,990 cents.

Both incorrect models returned 0 because their age predicates were faulty: North Mini used an invalid date-arithmetic calculation, while GPT-OSS matched only one exact birth date instead of the full interval for customers aged exactly 25.

The voting analysis also shows why a majority result must be checked against an independent reference: the fixed 2-of-3 panel accepts the wrong answer 0, whereas no answer reaches the 3-vote threshold in the full 3-of-5 panel.